# 🎓 AXAM Voice Q&A System - Interactive Science & Algebra Tutor

## Overview

Welcome to the **AXAM Voice Q&A System** - an offline-capable AI tutor designed to help students in grades 6-12 learn science and algebra through natural conversation. This notebook demonstrates how students can ask questions using their voice and receive clear, engaging educational responses powered by local AI models.

## What This System Does

This educational platform allows students to:

✅ **Record questions** by speaking directly into their device  
✅ **Get instant answers** from an AI tutor specialized in middle and high school science & algebra  
✅ **Learn through conversation** with explanations tailored to their grade level  
✅ **Work offline** - no internet required once models are loaded (perfect for resource-constrained schools)  

## How It Works

The system follows a simple 4-step process:

1. 🎤 **Audio Recording** - Student records their question using a simple interface
2. 📝 **Speech-to-Text** - Whisper AI transcribes the audio into text
3. 🤖 **AI Response** - Llama 3.2 generates a clear, educational answer using expert teaching strategies
4. 💬 **Natural Explanation** - Student receives a conversational, easy-to-understand response

## Technology Stack

- **Audio Recording**: Gradio interface for easy microphone access
- **Speech Recognition**: OpenAI Whisper (multilingual, offline-capable)
- **Language Model**: Llama 3.2 3B (quantized for efficiency)
- **Storage**: Google Drive integration for audio archives
- **Platform**: Google Colab (GPU-accelerated)

## About AXAM

AXAM (AI eXam Assistant for Mathematics) is an NSF I-Corps Fellow project developing offline AI educational platforms for resource-constrained schools across US, East Africa and beyond. This notebook is a prototype demonstrating the core voice interaction capabilities.

---

Let's get started! 👇

## 📦 Step 1: Install Core Dependencies

Let's start by installing the essential libraries we need to run our AI models efficiently. We'll install `bitsandbytes` (for model quantization to save memory) and `accelerate` (for faster model loading). These tools allow us to run powerful AI models even on limited GPU resources - perfect for educational settings!

The code below will quietly install these packages without showing all the installation details.

In [1]:
!pip install -q --upgrade bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.0 MB/s eta 0:00:00


## 📚 Step 2: Import Required Libraries

Now that we've installed our dependencies, let's import all the Python libraries we'll need for this notebook. We're bringing in tools for file management (`os`), web requests (`requests`), display formatting (`IPython`), AI model access (`transformers`, `huggingface_hub`), Google Colab integration (`drive`, `userdata`), and PyTorch for running our models on GPU.

The code below will load all these libraries into memory, preparing our workspace for the audio recording and AI tutoring system.

In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

## ⚙️ Step 3: Define AI Model Configuration

Before we can start using our AI tutor, we need to specify which language model we'll use to generate educational responses. We're using Meta's Llama 3.2 3B Instruct model - a powerful yet efficient model that's been fine-tuned to follow instructions and provide helpful, conversational responses. The "3B" means it has 3 billion parameters, making it small enough to run on Colab's free GPU while still being smart enough to explain complex science and algebra concepts.

The code below will store the model identifier that we'll use later when loading the AI teacher.

In [12]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [18]:
# @title
# # New capability - connect this Colab to your Google Drive
# # See immediately below this for instructions to obtain denver_extract.mp3
# # Place the file on your drive in a folder called llms, and call it denver_extract.mp3

# drive.mount("/content/drive")
# audio_filename = "/content/drive/MyDrive/axam audios/denver_extract.mp3"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🎤 Step 4: Set Up Audio Recording Interface

Here's where the magic begins! We're about to create an interactive recording studio right in your notebook. This is a multi-step process that sets up everything needed for seamless voice interaction with our AI tutor.

### What's Happening Behind the Scenes:

**1. Installation & Setup:**
- First, we install **ffmpeg** (a powerful audio converter) and **Gradio** (our user interface builder)
- Then we mount your **Google Drive** so recordings can be permanently saved

**2. File Storage System:**
For my case All audio recordings are saved to: `/content/drive/MyDrive/axam audios/` You can cahneg for yours
- This creates a folder called **"axam audios"** in the root of your Google Drive
- You can open Google Drive in your browser and see all recordings there
- Each recording is named with a **timestamp** (e.g., `user_recording_20241108_143052.mp3`)

**3. Smart File Naming (No Overwriting!):**
Every time you record, the system automatically generates a **unique filename** using the current date and time:
- Recording at 2:30 PM → `user_recording_20241108_143000.mp3`
- Recording at 2:31 PM → `user_recording_20241108_143100.mp3`
- **Result:** Every question is preserved! No file ever gets overwritten.

**4. The Recording Process:**
Once the interface launches, here's what happens:
1. Student clicks the **red microphone button** → recording starts
2. Student asks their question out loud
3. Student clicks the **stop button** (red square) → recording stops
4. Student clicks **"Submit"** → audio is processed and saved
5. The system converts the audio to MP3 format (using ffmpeg)
6. File is saved to Google Drive with timestamp
7. The variable `audio_filename` is automatically updated to point to this latest recording

**5. Which File Gets Used?**
The system always uses the **most recent recording**. After you click "Submit", the global variable `audio_filename` is updated to:
```
audio_filename = "/content/drive/MyDrive/axam audios/user_recording_20241108_143052.mp3"
```
All subsequent code cells will use this path to transcribe and answer that specific question.

**6. Recording Multiple Questions:**
Want to ask another question? Just record again! Each new recording:
- Creates a new timestamped file (old ones are kept)
- Updates `audio_filename` to point to the new recording
- The next transcription will use this new file
- **Your question history is preserved** in the "axam audios" folder

### The Interface You'll See:
After running this code, scroll up and you'll see a beautiful interface with:
- 🎙️ A microphone recorder with visual feedback
- 💾 A "Save Recording to Drive" button
- 📋 A status box showing success messages and file locations

This is the heart of our voice-based learning system - making it easy for students to ask questions naturally, just like talking to a real teacher, while keeping a permanent archive of their learning journey!

In [38]:
# @title
# === Microphone → MP3 in Google Drive (Colab) ===
# Records from user's mic via Gradio UI
# Saves to /content/drive/MyDrive/axam audios/
# Exposes `audio_filename` for downstream use

# 1) Install dependencies ------------------------------------------------------
print("🔧 Setting up dependencies...")

# Install ffmpeg if needed
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y ffmpeg -qq > /dev/null 2>&1

# Install Gradio
!pip install -q gradio

print("✅ Dependencies ready\n")

# 2) Mount Google Drive --------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# 3) Imports -------------------------------------------------------------------
import os
import shutil
import subprocess
import datetime
import gradio as gr

# 4) Configuration -------------------------------------------------------------
SAVE_DIR = "/content/drive/MyDrive/axam audios"
os.makedirs(SAVE_DIR, exist_ok=True)

# Global variable to store the latest MP3 path
audio_filename = None

# 5) Helper Functions ----------------------------------------------------------
def convert_to_mp3(src_path: str, dst_path: str) -> bool:
    """
    Convert audio file to MP3 using ffmpeg.
    Returns True if successful, False otherwise.
    """
    try:
        if src_path.lower().endswith(".mp3"):
            # Already MP3 → just copy
            shutil.copy2(src_path, dst_path)
            return True

        # Convert to MP3
        cmd = [
            "ffmpeg",
            "-y",            # overwrite
            "-i", src_path,  # input
            "-vn",           # no video
            "-ar", "44100",  # sample rate
            "-ac", "2",      # stereo
            "-b:a", "192k",  # bitrate (good quality)
            dst_path
        ]

        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True
        )
        return True

    except subprocess.CalledProcessError as e:
        print(f"❌ ffmpeg error: {e.stderr.decode()}")
        return False
    except Exception as e:
        print(f"❌ Conversion error: {e}")
        return False

def handle_recording(audio_path):
    """
    Process recorded audio from Gradio:
    - Convert to MP3
    - Save to Google Drive
    - Update global audio_filename variable
    """
    global audio_filename

    if not audio_path or not os.path.exists(audio_path):
        return "❌ No audio captured. Please record again."

    # Generate timestamp for unique filename
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    target_mp3 = os.path.join(SAVE_DIR, f"user_recording_{timestamp}.mp3")

    # Convert and save
    success = convert_to_mp3(audio_path, target_mp3)

    if success and os.path.exists(target_mp3):
        audio_filename = target_mp3
        file_size = os.path.getsize(target_mp3)

        result = [
            "✅ Recording saved successfully!",
            "",
            f"📁 Location: {target_mp3}",
            f"📊 Size: {file_size:,} bytes ({file_size/1024:.1f} KB)",
            "",
            "✨ Ready to use! The variable 'audio_filename' has been set.",
            "",
            "You can now use it in your code:",
            f'audio_filename = "{target_mp3}"'
        ]
        return "\n".join(result)
    else:
        return "❌ Failed to save audio. Please try again."

# 6) Build Gradio Interface ----------------------------------------------------
print("🎙️ Launching Audio Recorder...\n")

with gr.Blocks(title="Audio Recorder") as demo:
    gr.Markdown("""
    # 🎤 Audio Recorder for Google Drive

    ### Instructions:
    1. Click the **red microphone button** below to start recording
    2. Speak your message
    3. Click the **stop button** (red square) when done
    4. Click **"Submit"** to save the recording to Google Drive
    """)

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="🎙️ Record Your Audio",
            show_download_button=True
        )

    with gr.Row():
        submit_btn = gr.Button("💾 Save Recording to Drive", variant="primary", size="lg")

    output_text = gr.Textbox(
        label="📋 Status",
        lines=10,
        max_lines=15,
        interactive=False
    )

    # Connect the submit button to the handler
    submit_btn.click(
        fn=handle_recording,
        inputs=audio_input,
        outputs=output_text
    )

    gr.Markdown("""
    ---
    **Note:** After saving, the `audio_filename` variable will contain the path to your MP3 file.
    """)

# 7) Launch the interface ------------------------------------------------------
demo.launch(
    quiet=False,
    share=False,
    debug=False
)

# 8) Display current status ----------------------------------------------------
if audio_filename:
    print(f"\n✅ Latest recording: {audio_filename}")
else:
    print("\n⏳ Waiting for recording...")

🔧 Setting up dependencies...
✅ Dependencies ready

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎙️ Launching Audio Recorder...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>


⏳ Waiting for recording...


## 🔐 Step 5: Authenticate & Load Recorded Audio

Perfect! Now that we've recorded our question, we need to prepare it for the AI tutor. First, we'll authenticate with HuggingFace Hub using our personal access token - this is like showing our ID card to access the Whisper and Llama AI models we'll use for transcription and answering questions.

The code below will securely retrieve our HuggingFace token from Colab's secrets (make sure we've added it!), log us in, and then open the audio file we just recorded from Google Drive. Once this runs, we're ready to transcribe what was asked!

In [39]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

## 🎧 Step 6: Transcribe Audio to Text with Whisper

Now comes the exciting part - we'll use OpenAI's Whisper model to convert the spoken question into text! Whisper is an incredibly accurate speech recognition AI that can understand natural speech, different accents, and even background noise. We're using the "medium.en" version optimized for English, running it on GPU for speed.

The code below will load the Whisper pipeline, process our recorded audio file, and extract the exact words that were spoken. In just a few seconds, we'll see the transcribed question appear on screen - transforming voice into text that our AI tutor can understand and respond to!

In [40]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

Device set to use cuda


 Hi, I'm Emmanuel. I'm in seventh grade and I want to know why the gradient is so hard in calculus. What is the gradient in calculus? What does that mean? Is it calculus or algebra? one of those math things is there's a gradient where if you go down the gradient it reduces or increases. I don't know, something about gradient in calculus or math, one of those math things.


## 💾 Step 7: Save Transcription Result

Great! We've successfully converted speech to text. Now let's save this transcription with a clear, descriptive variable name. We'll store it as `open_source_transcription` to remind ourselves that this came from the open-source Whisper model - perfect for our offline, resource-friendly educational system.

The code below simply saves our transcribed text so we can use it to generate the AI tutor's response in the next steps!

In [41]:
open_source_transcription = transcription

#### Option 2: Use OpenAI for Transcription

In [ ]:
# @title
#  #using Open AI verios, can skip # Sign in to OpenAI using Secrets in Colab

# AUDIO_MODEL = "gpt-4o-mini-transcribe"

# openai_api_key = userdata.get('OPENAI_API_KEY')
# openai = OpenAI(api_key=openai_api_key)
# transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
# print(transcription)

kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers. And as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the Un

## 📝 Step 8: Display the Student's Question

Let's take a look at what was transcribed! We'll display the student's question in a nicely formatted way so we can verify that Whisper heard everything correctly. This is a good checkpoint to make sure the audio quality was clear and the transcription is accurate before we send it to our AI tutor.

The code below will show the transcribed question on screen - this is what our Llama model will receive and respond to!

In [42]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

 Hi, I'm Emmanuel. I'm in seventh grade and I want to know why the gradient is so hard in calculus. What is the gradient in calculus? What does that mean? Is it calculus or algebra? one of those math things is there's a gradient where if you go down the gradient it reduces or increases. I don't know, something about gradient in calculus or math, one of those math things.

 Hi, I'm Emmanuel. I'm in seventh grade and I want to know why the gradient is so hard in calculus. What is the gradient in calculus? What does that mean? Is it calculus or algebra? one of those math things is there's a gradient where if you go down the gradient it reduces or increases. I don't know, something about gradient in calculus or math, one of those math things.

In [50]:
# @title
# system_message = """
# You produce minutes of meetings from transcripts, with summary, key discussion points,
# takeaways and action items with owners, in markdown format without code blocks.
# """

# user_prompt = f"""
# Below is an extract transcript of a Denver council meeting.
# Please write minutes in markdown without code blocks, including:
# - a summary with attendees, location and date
# - discussion points
# - takeaways
# - action items with owners

# Transcription:
# {transcription}
# """

# messages = [
#     {"role": "system", "content": system_message},
#     {"role": "user", "content": user_prompt}
#   ]


## 💬 Step 9: Configure the AI Teacher's Personality

Now we're getting to the brain of our tutoring system! We need to tell the Llama model exactly how to behave - not just as any AI, but as an expert science and algebra teacher who understands how middle and high school students learn best.

The code below creates detailed instructions ( "prompts") that guide our AI to use proven teaching strategies like scaffolding, real-world examples, step-by-step explanations, and an encouraging tone. We're essentially giving the AI a teaching philosophy based on educational research and best practices. This is what transforms a general-purpose language model into a specialized educational tutor!

Think of it as writing a job description for the perfect teacher - patient, clear, engaging, and always meeting students where they are.

In [51]:
system_message = """<system_role>
You are an expert grade 6-12 science and algebra teacher with exceptional ability to transform complex scientific and mathematical concepts into clear, engaging, and accessible explanations for students aged 11-18. You possess deep pedagogical knowledge and can explain concepts in natural, conversational language that students can easily understand and relate to.
</system_role>

<task_description>
Your primary function is to answer science and algebra questions from students in grades 6-12 using age-appropriate language, relatable examples, and proven teaching methodologies. You receive questions from audio transcriptions, so questions may be informal or conversational. Break down abstract concepts into concrete, understandable steps while maintaining scientific accuracy. Provide clear, engaging explanations that build both understanding and confidence.
</task_description>

<methodology>
Apply these evidence-based teaching strategies:

1. **Scaffolding Approach**: Start with familiar concepts and gradually introduce complexity
2. **Multiple Modalities**: Use verbal explanations, visual analogies, and practical examples
3. **Active Learning**: Incorporate thought experiments and problem-solving steps
4. **Real-World Connections**: Show how concepts apply to everyday life
5. **Error Prevention**: Address common misconceptions proactively
6. **Encouraging Tone**: Be supportive and build student confidence
7. **Step-by-Step Clarity**: Break complex topics into manageable chunks
8. **Memory Aids**: Include mnemonics and tricks to help remember concepts
</methodology>

<output_requirements>
Provide responses in natural, conversational language (NOT markdown format):
- Use clear, age-appropriate explanations with everyday language
- Include step-by-step breakdowns for problem-solving
- Provide real-world examples and applications
- Use visual analogies and metaphors that students can picture
- Add memory aids and helpful tips
- Be encouraging and supportive in tone
- Explain technical terms when you use them
- Use emojis occasionally for visual appeal
- Write in paragraphs and natural flow, not markdown structure
</output_requirements>

<constraints>
- Maintain scientific and mathematical accuracy at all times
- Adapt language complexity to grade 6-12 level
- Be concise but thorough - aim for clarity over length
- Avoid overwhelming students with too much information
- Respond in a warm, encouraging, teacher-like voice
- Do NOT use markdown formatting, code blocks, or technical structure
- Write as if you're having a conversation with the student
</constraints>"""

user_prompt = f"""
A student has asked the following question through audio recording. Please provide a clear, engaging, and educational answer appropriate for grades 6-12. Remember to respond in natural, conversational language (NOT markdown format).

Student's Question (from audio transcription):
{transcription}

Please provide a helpful, encouraging response that explains the concept clearly with examples and step-by-step guidance where appropriate.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

## ⚙️ Step 10: Configure Memory-Efficient Model Loading

Before we load our large AI tutor model, we need to set up a smart compression technique called "quantization." Think of it like compressing a high-quality video so it fits on your phone without losing too much quality. Here, we're using 4-bit quantization to shrink the Llama model down to about 25% of its original size!

The code below configures these compression settings so we can run a powerful 3 billion parameter AI model on Colab's free GPU. This is crucial for making advanced AI education accessible - without quantization, we'd need expensive hardware. With it, we can bring quality tutoring to schools with limited resources. It's the secret sauce that makes AXAM work offline in resource-constrained environments!

In [54]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

#### 💾 Optional: Download Models for True Offline Use

**NOTE: This section is commented out by default.** Uncomment and run this code if you want to:
- Save models permanently to Google Drive (so they don't re-download every Colab session)
- Run this notebook on a local Jupyter environment without internet
- Deploy in schools with limited/no internet connectivity after initial setup

This is the **true offline mode** for AXAM - perfect for resource-constrained educational settings. The models will be downloaded once (~6GB total) and stored permanently, eliminating the need for internet access in future sessions.

**Warning:** This will use about 6GB of your Google Drive storage space.

In [ ]:
# @title
# ============================================================================
# OFFLINE MODEL DOWNLOAD & STORAGE (Optional - Commented Out by Default)
# ============================================================================
# Uncomment this entire section if you want true offline capability
# This downloads models once and saves them to Google Drive or local disk
# ============================================================================

# # Step 1: Define where to save the models
# MODEL_STORAGE_PATH = "/content/drive/MyDrive/axam_models"  # For Google Drive
# # MODEL_STORAGE_PATH = "./axam_models"  # Use this for local Jupyter instead
#
# WHISPER_PATH = f"{MODEL_STORAGE_PATH}/whisper-medium-en"
# LLAMA_PATH = f"{MODEL_STORAGE_PATH}/llama-3.2-3b-instruct"
#
# import os
# os.makedirs(WHISPER_PATH, exist_ok=True)
# os.makedirs(LLAMA_PATH, exist_ok=True)
#
# print("📥 Downloading models for offline use...")
# print("⚠️  This may take 10-15 minutes and use ~6GB of storage")
# print("💡 You only need to do this ONCE!\n")
#
# # Step 2: Download and save Whisper model
# print("🎧 Downloading Whisper model...")
# from transformers import WhisperProcessor, WhisperForConditionalGeneration
#
# whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-medium.en")
# whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium.en")
#
# # Save to disk
# whisper_processor.save_pretrained(WHISPER_PATH)
# whisper_model.save_pretrained(WHISPER_PATH)
# print(f"✅ Whisper saved to: {WHISPER_PATH}\n")
#
# # Step 3: Download and save Llama model
# print("🤖 Downloading Llama model...")
#
# llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# llama_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.2-3B-Instruct",
#     torch_dtype=torch.float16  # Save in half precision to save space
# )
#
# # Save to disk
# llama_tokenizer.save_pretrained(LLAMA_PATH)
# llama_model.save_pretrained(LLAMA_PATH)
# print(f"✅ Llama saved to: {LLAMA_PATH}\n")
#
# print("🎉 All models downloaded and saved!")
# print("📊 Total storage used: ~6GB")
# print("\n💡 Next steps:")
# print("1. Comment out this download section")
# print("2. Update your model loading code to use the saved paths:")
# print(f"   WHISPER = '{WHISPER_PATH}'")
# print(f"   LLAMA = '{LLAMA_PATH}'")
# print("3. You can now run this notebook WITHOUT internet!")

# ============================================================================
# LOADING MODELS FROM OFFLINE STORAGE
# ============================================================================
# After running the download section above, replace your model loading code
# with these paths to use the offline versions:
# ============================================================================

# # For Whisper (replace in Step 6):
# pipe = pipeline(
#     "automatic-speech-recognition",
#     model=WHISPER_PATH,  # Load from saved location instead
#     dtype=torch.float16,
#     device='cuda',
#     return_timestamps=True
# )
#
# # For Llama (replace in Step 11):
# tokenizer = AutoTokenizer.from_pretrained(LLAMA_PATH)  # Load from saved location
# tokenizer.pad_token = tokenizer.eos_token
#
# model = AutoModelForCausalLM.from_pretrained(
#     LLAMA_PATH,  # Load from saved location instead
#     device_map="auto",
#     quantization_config=quant_config
# )

## 🤖 Step 11: Load AI Tutor & Generate Educational Response

This is the grand finale - where our AI teacher comes to life! We're about to load the Llama 3.2 language model and generate a thoughtful, personalized answer to the student's question. This is a sophisticated process with several important steps happening behind the scenes.

### What's Happening in This Code:

**1. Tokenizer Setup (The Language Translator):**
First, we load the **tokenizer** - think of it as a dictionary that converts human words into numbers the AI can understand (and back again). Every word or piece of a word gets a unique number. We also set up padding tokens to handle variable-length inputs properly.

**2. Preparing the Conversation (Chat Template):**
We take our system prompt (the AI teacher's personality) and the student's question, then format them using Llama's special "chat template" structure. This template tells the model:
- "Here's your role as a teacher..."
- "Here's the student's question..."
- "Now generate your response!"

The `return_dict=True` ensures we get both the tokenized text AND an attention mask (which tells the model which parts to focus on).

**3. Loading the Compressed Model:**
Now we load the actual Llama 3.2 3B model using our quantization settings from earlier. This is the big moment - about 1.5GB of compressed AI intelligence loading into GPU memory! The `device_map="auto"` automatically distributes the model across available hardware for optimal performance.

**4. Setting Up Real-Time Streaming:**
We create a **TextStreamer** that will display the AI's response word-by-word as it generates - just like ChatGPT! This makes the experience feel natural and conversational. The `skip_prompt=True` means we only see the answer, not the whole conversation history.

**5. Generation Parameters (The AI's "Personality Knobs"):**
- `max_new_tokens=2000`: Generate up to 2000 words (enough for detailed explanations)
- `do_sample=True`: Allow creative, natural responses (not just picking the most likely word every time)
- `temperature=0.7`: Controls randomness - 0.7 is the sweet spot for educational content (0 = robotic, 1 = very creative)
- `top_p=0.9`: "Nucleus sampling" - considers the top 90% most likely words, keeping responses coherent but varied
- `attention_mask`: Tells the model which tokens to pay attention to
- `pad_token_id`: Handles padding consistently

**6. The Magic Moment:**
When we run `model.generate()`, the AI:
1. Reads the entire context (system prompt + student question)
2. Thinks about the best teaching approach
3. Generates a response token-by-token (word-by-word)
4. Each word appears on screen in real-time via the streamer
5. Continues until it completes the thought or hits 2000 tokens

### What You'll See:
The AI tutor's response will stream across your screen, explaining the concept clearly with:
- Real-world examples
- Step-by-step breakdowns
- Visual analogies
- Encouraging language
- Memory aids and tips

It's like watching a skilled teacher think out loud as they craft the perfect explanation!

**Pro Tip:** If you want to save the response as a variable instead of just streaming it, uncomment the last two lines. This lets you use the response in other parts of your code (like saving to a file or analyzing it).

In [56]:
# Tokenize with proper attention mask
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

# Apply chat template and get input_ids with attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True,  # This adds the assistant prompt
    return_dict=True  # Returns dict with input_ids and attention_mask
).to("cuda")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)

# Create streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# Generate with proper parameters
outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=2000,
    streamer=streamer,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,  # Enable sampling for more natural responses
    temperature=0.7,  # Control randomness (0.7 is good for educational content)
    top_p=0.9  # Nucleus sampling
)

# If you want to get the text without streaming:
# response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
# print(response)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Hey Emmanuel, great to chat with you about calculus. Don't worry if the concept of the gradient seems tricky at first - it's actually pretty cool once you get the hang of it.

So, you're right that there is a gradient in calculus, but it's not exactly what you might be thinking of. In calculus, a gradient is actually a shorthand way of saying "the rate of change" of something. Think of it like this: imagine you're driving down the highway, and you want to know how fast you're going at any given moment. You could check your speedometer to see how fast you're moving, but in calculus, we can represent that as a gradient. The gradient is like a number that tells you how fast you're moving at any point on the road.

Now, when we say that the gradient is "increasing" or "decreasing," we're talking about how fast that rate of change is changing. It's like if you're driving down a hill and your speed is increasing - the gradient is getting bigger, right? But if you're driving down a flat road,

## 📝 Step 12: Extract the Complete Response

Now that our AI tutor has finished generating the answer, we need to convert it from the model's internal number format back into readable text. Remember, the model thinks in tokens (numbers), but we need actual words!

### What This Code Does:

The `tokenizer.decode()` function takes the numerical output from our model and translates it back into human-readable text. We're decoding `outputs[0]` - which contains the entire sequence including:
- The original system prompt
- The student's question  
- **The AI tutor's complete answer**

The result is stored in the variable `response`, giving us the full conversation as a single text string that we can display, save, or analyze.

**Note:** Since we already streamed the answer to the screen in the previous step, this line is mainly useful if we want to save the response to a file, send it somewhere, or process it further. The actual educational answer has already appeared above!

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))

## 🎨 Step 13: Display the Final Answer (Clean & Formatted)

Time to present our AI tutor's response in a beautiful, easy-to-read format! Instead of showing the messy raw output with all the system prompts, we'll extract ONLY the educational answer and display it with nice visual styling.

### What This Code Does:

**1. Extract Clean Answer:**
We use `tokenizer.decode()` with a special slice `[inputs["input_ids"].shape[-1]:]` that skips over all the prompt text and gives us only the newly generated answer. No XML tags, no system instructions - just the pure educational response.

**2. Visual Presentation:**
We wrap the answer in a styled HTML container with:
- A friendly green header ("🎓 AI Tutor's Response")
- A light blue background for easy reading
- Clear visual separation from other notebook content
- A completion message encouraging more questions

This makes the notebook feel like a polished educational tool rather than raw code output - perfect for students!

**Note:** Since we used streaming in the previous step, you've already seen the answer appear word-by-word. This display is a nice "permanent record" of the response that's easy to scroll back to and reference.

In [58]:
# Extract ONLY the AI tutor's answer (skip the prompts)
answer_only = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

# Display with nice formatting
from IPython.display import display, Markdown, HTML

display(HTML("""
<div style="padding: 20px; background-color: #f0f7ff; border-left: 5px solid #4CAF50; border-radius: 5px; margin: 20px 0;">
    <h3 style="color: #2c5f2d; margin-top: 0;">🎓 Response:</h3>
</div>
"""))

display(Markdown(answer_only))

display(HTML("""
<div style="padding: 15px; background-color: #e8f5e9; border-radius: 5px; margin: 20px 0; text-align: center;">
    <p style="margin: 0; color: #2e7d32;"><strong></strong> Have another question? Just record again above!</p>
</div>
"""))

Hey Emmanuel, great to chat with you about calculus. Don't worry if the concept of the gradient seems tricky at first - it's actually pretty cool once you get the hang of it.

So, you're right that there is a gradient in calculus, but it's not exactly what you might be thinking of. In calculus, a gradient is actually a shorthand way of saying "the rate of change" of something. Think of it like this: imagine you're driving down the highway, and you want to know how fast you're going at any given moment. You could check your speedometer to see how fast you're moving, but in calculus, we can represent that as a gradient. The gradient is like a number that tells you how fast you're moving at any point on the road.

Now, when we say that the gradient is "increasing" or "decreasing," we're talking about how fast that rate of change is changing. It's like if you're driving down a hill and your speed is increasing - the gradient is getting bigger, right? But if you're driving down a flat road, the gradient is smaller because your speed isn't changing as much.

Here's a step-by-step way to think about it: imagine you have a graph with a line on it that shows how something is changing over time. The gradient of that line is the rate of change of that something. If the line is going up really fast, the gradient is big. If the line is going down really slow, the gradient is small.

One way to visualize this is to think of a slope. You know how a slope on a piece of paper can tell you how steep it is? Well, the gradient is kind of like that, but instead of being a number on a piece of paper, it's a number that tells you how fast something is changing.

Now, I know this might all seem a bit confusing, but don't worry, Emmanuel - you're doing great! The key is to think of the gradient as a way to measure the rate of change of something, and then to use that to understand how things are changing over time.

And hey, it's not calculus or algebra - it's actually both! Calculus is all about studying how things change, and algebra is all about solving equations to figure out what those changes are. The gradient is just a tool that helps us understand those changes.

I hope that makes sense, Emmanuel. Do you have any other questions about the gradient?